# Empirical Results Evaluation

The results in this file were computed using [R version 4.5.3](https://cran.r-project.org/src/base/R-4/) and Python version 3.10.7 with the packages and versions listed in `requirements.txt`.

In [1]:
import numpy as np
import pandas as pd          
import statsmodels.api as sm 
import rpy2.robjects as ro
from rpy2.robjects import numpy2ri
from scipy import stats

# Load CSV
df = pd.read_csv('results.csv', keep_default_na=False)
df = df.sort_values(by='name')

## Project Filtering (Table 1)

In [2]:
# consider GCP and Azure projects as one (both marketplaces)
df['source'] = df['source'].replace('Awesome, Azure', 'Awesome, GCP_Azure')
df['source'] = df['source'].replace('GCP, Azure', 'GCP_Azure')
df['source'] = df['source'].replace('Azure', 'GCP_Azure')
df['source'] = df['source'].replace('GCP', 'GCP_Azure')

# Split entries that contain multiple TEEs
df['tee_type'] = df['tee_type'].astype(str)
df['tee_type'] = df['tee_type'].str.split(',')
df = df.explode('tee_type')
df['tee_type'] = df['tee_type'].str.strip()

df_split_source = df.copy(deep=True)

# split by source to consider them independently
df_split_source['source'] = df_split_source['source'].astype(str)
df_split_source['source'] = df_split_source['source'].str.split(',')
df_split_source = df_split_source.explode('source')
df_split_source['source'] = df_split_source['source'].str.strip()

# remove projects that consider TEEs, we don't look at (e.g., TrustZone)
df_split_source = df_split_source[df_split_source['tee_type'] != 'Other']

# initial candidate set
init_awesome = len(df_split_source[df_split_source['source'] == 'Awesome'])
init_market = len(df_split_source[df_split_source['source'] == 'GCP_Azure'])

# TEE integrated (C1)
tee_integrated_awesome = len(df_split_source[(df_split_source['source'] == 'Awesome') & (df_split_source['tee_type'] != '')])
tee_integrated_market = len(df_split_source[(df_split_source['source'] == 'GCP_Azure') & (df_split_source['tee_type'] != '')])

# directly purchasable (C2)
purchasable_awesome = 'NA'
purchasable_market = len(df_split_source[(df_split_source['source'] == 'GCP_Azure') & (df_split_source['purchasable'] == 'Yes')])

# Active (C3)
active_awesome = len(df_split_source[(df_split_source['source'] == 'Awesome') & (df_split_source['activity'] == 'active commit') & (df_split_source['stability'] != 'instable')])
active_market = len(df_split_source[(df_split_source['source'] == 'GCP_Azure') & (df_split_source['activity'] == 'active commit') & (df_split_source['stability'] != 'instable')])

# final dataset (C1-C3)
final_awesome = len(df_split_source[(df_split_source['source'] == 'Awesome') & (~df_split_source['activity'].isin(['inactive commit', 'archived', 'invalid link', 'discontinued'])) & (df_split_source['stability'] != 'instable') & (df_split_source['tee_type'] != '')])
final_market = len(df_split_source[(df_split_source['source'] == 'GCP_Azure') & (~df_split_source['activity'].isin(['inactive commit', 'archived', 'invalid link', 'discontinued'])) & (df_split_source['stability'] != 'instable') & (df_split_source['tee_type'] != '') & (df_split_source['purchasable'] == 'Yes')])

# print table
print('\\textbf{Stage} & \\textbf{Awesome SGX} & \\textbf{Marketplaces} \\\\')
print(f'Initial candidate set & {init_awesome} & {init_market} \\\\')
print(f'TEE-integrated (C1) & {tee_integrated_awesome} & {tee_integrated_market} \\\\')
print(f'Directly purchasable (C2) & {purchasable_awesome} & {purchasable_market} \\\\')
print(f'Active (C3) & {active_awesome} & {active_market} \\\\')
print(f'Final dataset & {final_awesome} & {final_market} \\\\')

print(f"\nSGX only in Awesome SGX: {len(df_split_source[(df_split_source['source'] == 'Awesome') & (df_split_source['tee_type'] == 'SGX')])}")

\textbf{Stage} & \textbf{Awesome SGX} & \textbf{Marketplaces} \\
Initial candidate set & 232 & 35 \\
TEE-integrated (C1) & 203 & 27 \\
Directly purchasable (C2) & NA & 29 \\
Active (C3) & 122 & 35 \\
Final dataset & 95 & 21 \\

SGX only in Awesome SGX: 180


**Filter df according to C1-C3**

In [3]:
def filter_c1c3(frame):
    # Remove entries for TEEs other than SGX/TDX/SEV-SNP
    frame = frame[frame['tee_type'] != 'Other']

    # Remove projects that are not TEE-integrated: Keep only rows where TEE is not empty / NaN
    frame = frame[frame['tee_type'].notna() & (frame['tee_type'].str.strip() != '')]

    # remove marketplace projects that are not purchasable
    frame = frame[frame['purchasable'] != 'No']

    # Remove inactive projects
    frame = frame[frame['activity'] == 'active commit']

    # Remove unstable projects
    frame = frame[frame['stability'] != 'instable']

    return frame

df = df.reset_index(drop=True)

df = filter_c1c3(df)

## Dataset Overview (Table 2)

In [4]:
TEES = ['SGX', 'SEV-SNP', 'TDX', 'None']

# Final dataset
print('\\textbf{{Dataset}} & & & & \\\\')
line = 'Total projects '
for tee in TEES:
    num = len(df[df['tee_type'] == tee])
    line += f"& {num} "
line += '\\\\'
print(line)
print('\\hline')

# project type
print('\\textbf{{Project type}} & & & & \\\\')
line = '\\rule{1em}{0pt} Commercial '
for tee in TEES:
    num = len(df[(df['tee_type'] == tee) & (df['project_type'] == 'commercial')])
    line += f"& {num} "
line += '\\\\'
print(line)

line = '\\rule{1em}{0pt} Academic '
for tee in TEES:
    num = len(df[(df['tee_type'] == tee) & (df['project_type'] == 'academic')])
    line += f"& {num} "
line += '\\\\'
print(line)
print('\\hline')

# build environment
ENVS = ['None', 'Docker file', 'Docker image', 'Yocto', 'Bazel', 'Nix']
print('\\textbf{{Primary build system}} & & & & \\\\')
for env in ENVS:
    line = f'\\rule{{1em}}{{0pt}} {env} '
    for tee in TEES:
        num = len(df[(df['tee_type'] == tee) & (df['build_system'] == env)])
        line += f"& {num} "
    line += '\\\\'
    print(line)
print('\\hline')

# source code + reference
print('\\textbf{{Reference availability}} & & & & \\\\')
line = '\\rule{1em}{0pt} Source code only '
for tee in TEES:
    num = len(df[(df['tee_type'] == tee) & (df['source_code'] == 'Yes') & (df['build_instructions'] == 'Yes') & (df['reference'] == 'No')])
    line += f"& {num} "
line += '\\\\'
print(line)

line = '\\rule{1em}{0pt} Reference only '
for tee in TEES:
    num = len(df[(df['tee_type'] == tee) & (df['source_code'] == 'No') & (df['reference'] == 'Yes')])
    line += f"& {num} "
line += '\\\\'
print(line)

line = '\\rule{1em}{0pt} Source code + reference '
for tee in TEES:
    num = len(df[(df['tee_type'] == tee) & (df['source_code'] == 'Yes') & (df['build_instructions'] == 'Yes') & (df['reference'] == 'Yes')])
    line += f"& {num} "
line += '\\\\'
print(line)
print('\\hline')

# reproducibility
print('\\textbf{{Reproducibility}} & & & & \\\\')
line = '\\rule{1em}{0pt} Unreproducible '
for tee in TEES:
    num = len(df[(df['tee_type'] == tee) & (df['reproduced'] == 'No')])
    line += f"& {num} "
line += '\\\\'
print(line)

line = '\\rule{1em}{0pt} Reproducible guided scenario '
for tee in TEES:
    num = len(df[(df['tee_type'] == tee) & (df['reproduced'] == 'Yes (guided)')])
    line += f"& {num} "
line += '\\\\'
print(line)

line = '\\rule{1em}{0pt} Reproducible unguided scenario '
for tee in TEES:
    num = len(df[(df['tee_type'] == tee) & (df['reproduced'] == 'Yes (indep)')])
    line += f"& {num} "
line += '\\\\'
print(line)
print('\\hline')

\textbf{{Dataset}} & & & & \\
Total projects & 95 & 15 & 3 & 2 \\
\hline
\textbf{{Project type}} & & & & \\
\rule{1em}{0pt} Commercial & 65 & 14 & 3 & 2 \\
\rule{1em}{0pt} Academic & 30 & 1 & 0 & 0 \\
\hline
\textbf{{Primary build system}} & & & & \\
\rule{1em}{0pt} None & 74 & 8 & 1 & 2 \\
\rule{1em}{0pt} Docker file & 11 & 3 & 0 & 0 \\
\rule{1em}{0pt} Docker image & 8 & 1 & 0 & 0 \\
\rule{1em}{0pt} Yocto & 0 & 1 & 1 & 0 \\
\rule{1em}{0pt} Bazel & 1 & 1 & 1 & 0 \\
\rule{1em}{0pt} Nix & 1 & 1 & 0 & 0 \\
\hline
\textbf{{Reference availability}} & & & & \\
\rule{1em}{0pt} Source code only & 57 & 2 & 0 & 0 \\
\rule{1em}{0pt} Reference only & 1 & 0 & 0 & 0 \\
\rule{1em}{0pt} Source code + reference & 22 & 4 & 1 & 0 \\
\hline
\textbf{{Reproducibility}} & & & & \\
\rule{1em}{0pt} Unreproducible & 89 & 11 & 3 & 2 \\
\rule{1em}{0pt} Reproducible guided scenario & 3 & 0 & 0 & 0 \\
\rule{1em}{0pt} Reproducible unguided scenario & 3 & 4 & 0 & 0 \\
\hline


## Analysis Summary (Table 3)

In [5]:
df_ref = df.copy(deep=True)
df_ref = df_ref[(df_ref['source_code'] == 'Yes') & (df_ref['build_instructions'] == 'Yes') & (df_ref['reference'] == 'Yes')]

print('\\textbf{Project} & \\textbf{Built (Unguided)} & \\textbf{Reproduced (Unguided)} & \\textbf{Replied} & \\textbf{Built (Guided)} & \\textbf{Reproduced (Guided)} \\\\')

for tee in TEES:
    if tee == 'None':
        continue 

    print(f'\\multicolumn{{6}}{{|c|}}{{\\textbf{tee}}}\\\\')
    
    df_tee = df_ref.copy(deep=True)
    df_tee = df_tee[df_tee['tee_type'] == tee]
    df_tee.reset_index()

    for index, row in df_tee.iterrows():
        line = f" "
        build_unguided = '\\cmark' if row['compiled'] == 'Yes (indep)' else '\\xmark'
        reproduced_unguided = '\\cmark' if row['reproduced'] == 'Yes (indep)' else '\\xmark'

        if build_unguided == '\\cmark' and reproduced_unguided == '\\cmark': 
            replied = '--'
        else:
            replied = '\\cmark' if row['replied'] == 'Yes' else '\\xmark'

        if build_unguided == '\\cmark':
            build_guided = '--'
        else:
            build_guided = '\\cmark' if row['compiled'] == 'Yes (guided)' else '\\xmark'

        if reproduced_unguided == '\\cmark':
            reproduced_guided = '--'
        else:
            reproduced_guided = '\\cmark' if row['reproduced'] == 'Yes (guided)' else '\\xmark'

        print(f"{row['name']} & {build_unguided} & {reproduced_unguided} & {replied} & {build_guided} & {reproduced_guided} \\\\")

\textbf{Project} & \textbf{Built (Unguided)} & \textbf{Reproduced (Unguided)} & \textbf{Replied} & \textbf{Built (Guided)} & \textbf{Reproduced (Guided)} \\
\multicolumn{6}{|c|}{\textbfSGX}\\
CCF & \cmark & \xmark & \cmark & -- & \cmark \\
Date-in-use Encrypting MongoDB & \cmark & \xmark & \cmark & -- & \xmark \\
Edgeless RT & \cmark & \xmark & \xmark & -- & \xmark \\
Ego & \cmark & \cmark & -- & -- & -- \\
Google asylo & \cmark & \xmark & \xmark & -- & \xmark \\
Gramine & \cmark & \xmark & \cmark & -- & \cmark \\
Marblerun & \cmark & \xmark & \cmark & -- & \cmark \\
MobileCoin & \cmark & \xmark & \xmark & -- & \xmark \\
Mystikos & \cmark & \xmark & \xmark & -- & \xmark \\
Nginx-SGX & \cmark & \xmark & \cmark & -- & \xmark \\
Oasis Network & \cmark & \xmark & \xmark & -- & \xmark \\
Occlum & \cmark & \xmark & \xmark & -- & \xmark \\
Phala blockchain & \cmark & \xmark & \xmark & -- & \xmark \\
Private SSE Schemes & \xmark & \xmark & \xmark & \xmark & \xmark \\
Safeheron's TEE  & \xmark 

## Hypothesis Tests (Section 3.2)

In [6]:
# consider only non-academic projects for the hypothesis tests
df = df[df['project_type'] != 'academic']

**Build system:**

In [7]:
# H0: The probability of reproducibility is the same for all build types.

# observed data
none        = len(df[df['build_system'] == 'None'])
docker_f    = len(df[df['build_system'] == 'Docker file'])
docker_i    = len(df[df['build_system'] == 'Docker image'])
pintool     = len(df[df['build_system'].isin(['Bazel', 'Nix', 'Yocto'])])

none_r        = len(df[(df['build_system'] == 'None') & (df['reproduced'].isin(['Yes (guided)', 'Yes (indep)']))])
none_nr       = len(df[(df['build_system'] == 'None') & (df['reproduced'].isin(['No', 'NA']))])

docker_f_r    = len(df[(df['build_system'] == 'Docker file') & (df['reproduced'].isin(['Yes (guided)', 'Yes (indep)']))])
docker_f_nr   = len(df[(df['build_system'] == 'Docker file') & (df['reproduced'].isin(['No', 'NA']))])

docker_i_r    = len(df[(df['build_system'] == 'Docker image') & (df['reproduced'].isin(['Yes (guided)', 'Yes (indep)']))])
docker_i_nr   = len(df[(df['build_system'] == 'Docker image') & (df['reproduced'].isin(['No', 'NA']))])

pintool_r     = len(df[(df['build_system'].isin(['Bazel', 'Nix', 'Yocto'])) & (df['reproduced'].isin(['Yes (guided)', 'Yes (indep)']))])
pintool_nr    = len(df[(df['build_system'].isin(['Bazel', 'Nix', 'Yocto'])) & (df['reproduced'].isin(['No', 'NA']))])

contingency_table = [
    [none_r, docker_f_r, docker_i_r, pintool_r], 
    [none_nr, docker_f_nr, docker_i_nr, pintool_nr]
]

flat = [item for row in contingency_table for item in row]

ro.globalenv['m'] = ro.r.matrix(
    ro.IntVector(flat),
    nrow=2,
    byrow=True
)

result = ro.r('fisher.test(m)')
print(result)

print(contingency_table)


	Fisher's Exact Test for Count Data

data:  m
p-value = 0.01225
alternative hypothesis: two.sided


[[3, 2, 2, 3], [54, 9, 7, 4]]


**TEE type:**

In [8]:
# H0: The probability of reproducibility is the same for all TEE types.

# observed data
sgx = len(df[df['tee_type'] == 'SGX'])
sev = len(df[df['tee_type'] == 'SEV-SNP'])
tdx = len(df[df['tee_type'] == 'TDX'])

sgx_r    = len(df[(df['tee_type'] == 'SGX') & (df['reproduced'].isin(['Yes (guided)', 'Yes (indep)']))])
sgx_nr   = len(df[(df['tee_type'] == 'SGX') & (df['reproduced'].isin(['No', 'NA']))])

sev_r    = len(df[(df['tee_type'] == 'SEV-SNP') & (df['reproduced'].isin(['Yes (guided)', 'Yes (indep)']))])
sev_nr   = len(df[(df['tee_type'] == 'SEV-SNP') & (df['reproduced'].isin(['No', 'NA']))])

tdx_r    = len(df[(df['tee_type'] == 'TDX') & (df['reproduced'].isin(['Yes (guided)', 'Yes (indep)']))])
tdx_nr   = len(df[(df['tee_type'] == 'TDX') & (df['reproduced'].isin(['No', 'NA']))])

contingency_table = np.array([
    [sgx_r, sev_r, tdx_r], 
    [sgx_nr, sev_nr, tdx_nr]
])

flat = [item for row in contingency_table for item in row]

ro.globalenv['m'] = ro.r.matrix(
    ro.IntVector(flat),
    nrow=2,
    byrow=True
)

result = ro.r('fisher.test(m)')
print(result)

print(contingency_table)


	Fisher's Exact Test for Count Data

data:  m
p-value = 0.09903
alternative hypothesis: two.sided


[[ 6  4  0]
 [59 10  3]]
